# Phase B figure recovery
Load the existing Phase B results archive and existing checkpoints only. Do not retrain Phase B models. Recover segmentation predictions, ANFIS membership functions/response surface, or documented timing only when their original checkpoint configuration and feature scaling are available.

In [ ]:
# Recovery uses the original exports/checkpoints only; it deliberately does not invoke training.
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess, sys
uploaded=files.upload()
archives=[Path('/content')/n for n in uploaded if n.lower().endswith('.zip')]
if not archives: raise RuntimeError('Upload the completed Phase B results ZIP.')
work=Path('/content/phase_b_recovery'); shutil.rmtree(work,ignore_errors=True); work.mkdir()
with zipfile.ZipFile(archives[0]) as z: z.extractall(work)
matches=list(work.rglob('predictions_regression.csv'))
if not matches: raise RuntimeError('No saved regression predictions: recovery cannot manufacture them or retrain.')
results_root=matches[0].parents[1]
repo=Path('/content/cassava-navigation-ai')
if not (repo/'src/phase_B_visualization.py').exists(): subprocess.check_call(['git','clone','--depth','1','https://github.com/bechosen-spec/cassava-navigation-ai.git',str(repo)])
sys.path.insert(0,str(repo)); %pip -q install matplotlib scikit-learn pandas
from src.phase_B_visualization import generate_recovery_figures, write_recovery_reports, export_archives
entries=generate_recovery_figures(results_root,repo/'research_figures')
total=write_recovery_reports(entries,repo/'research_figures'); export_archives(results_root,repo/'research_figures')
print(f'Recovered {len(entries)} supplementary figures; {total} canonical PNG figures available.')
files.download(str(repo/'cassava_navigation_phase_B_results_complete.zip')); files.download(str(repo/'cassava_navigation_research_figures.zip'))
